In [0]:
from pyspark.sql.functions import (
    col,
    when,
    current_timestamp,
    year,
    month,
    dayofmonth
)

SILVER_SCHEMA = "workspace.silver"
GOLD_SCHEMA = "workspace.gold"

spark.sql(f"""
CREATE SCHEMA IF NOT EXISTS {GOLD_SCHEMA}
""")

In [0]:
dim_artista = (
    spark.table(f"{SILVER_SCHEMA}.artistas")
        .select(
            col("id").alias("artista_id"),
            col("nome").alias("nome_artista"),
            col("pais"),
            col("genero"),
            col("criado_em")
        )
        .withColumn("gold_created_at", current_timestamp())
)

(
    dim_artista.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(f"{GOLD_SCHEMA}.dim_artista")
)

In [0]:
dim_album = (
    spark.table(f"{SILVER_SCHEMA}.albuns")
        .select(
            col("id").alias("album_id"),
            col("artista_id"),
            col("titulo").alias("titulo_album"),
            col("ano_lancamento"),
            col("total_faixas")
        )
        .withColumn("gold_created_at", current_timestamp())
)

(
    dim_album.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(f"{GOLD_SCHEMA}.dim_album")
)

In [0]:
dim_musica = (
    spark.table(f"{SILVER_SCHEMA}.musicas")
        .select(
            col("id").alias("musica_id"),
            col("album_id"),
            col("titulo").alias("titulo_musica"),
            col("duracao_segundos"),
            col("numero_faixa")
        )
        .withColumn("gold_created_at", current_timestamp())
)

(
    dim_musica.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(f"{GOLD_SCHEMA}.dim_musica")
)

In [0]:
dim_usuario = (
    spark.table(f"{SILVER_SCHEMA}.usuarios")
        .select(
            col("id").alias("usuario_id"),
            col("nome").alias("nome_usuario"),
            col("email"),
            col("plano"),
            col("criado_em")
        )
        .withColumn("gold_created_at", current_timestamp())
)

(
    dim_usuario.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(f"{GOLD_SCHEMA}.dim_usuario")
)

In [0]:
reproducoes = spark.table(f"{SILVER_SCHEMA}.reproducoes")
musicas = spark.table(f"{SILVER_SCHEMA}.musicas")
albuns = spark.table(f"{SILVER_SCHEMA}.albuns")

In [0]:
fato_reproducao = (
    reproducoes
        .join(
            musicas,
            reproducoes.musica_id == musicas.id,
            "inner"
        )
        .join(
            albuns,
            musicas.album_id == albuns.id,
            "inner"
        )
        .select(
            reproducoes.id.alias("reproducao_id"),

            reproducoes.usuario_id,
            reproducoes.musica_id,

            musicas.album_id,
            albuns.artista_id,

            reproducoes.reproduzida_em,

            year(reproducoes.reproduzida_em).alias("ano"),
            month(reproducoes.reproduzida_em).alias("mes"),
            dayofmonth(reproducoes.reproduzida_em).alias("dia"),

            reproducoes.duracao_ouvida_segundos,
            musicas.duracao_segundos,

            when(
                reproducoes.duracao_ouvida_segundos >= musicas.duracao_segundos,
                True
            ).otherwise(False).alias("musica_completa"),

            current_timestamp().alias("gold_created_at")
        )
)

(
    fato_reproducao.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(f"{GOLD_SCHEMA}.fato_reproducao")
)

In [0]:
gold_tables = [
    "dim_artista",
    "dim_album",
    "dim_musica",
    "dim_usuario",
    "fato_reproducao"
]

print("Camada Gold criada com sucesso.\n")

In [0]:
for table in gold_tables:
    count = spark.table(f"{GOLD_SCHEMA}.{table}").count()
    print(f"{table}: {count} registros")